In [ ]:
import json
import os
import sys
import time

REPO_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for path in (REPO_ROOT, os.path.join(REPO_ROOT, "GUI")):
    if path not in sys.path:
        sys.path.insert(0, path)

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.hdmi_backend import AudioLabHdmiBackend
from audio_lab_pynq.hdmi_effect_state_mirror import HdmiEffectStateMirror
from pynq_multi_fx_gui import (
    AppState,
    THEMES,
    make_pynq_static_render_cache,
    render_frame_800x480,
)

HOLD_SECONDS_PER_STEP = 2.0
FINAL_HOLD_SECONDS = 30
RETURN_TO_SAFE_BYPASS = False
THEME = "pipboy-green" if "pipboy-green" in THEMES else None
VARIANT = "compact-v2"
OFFSET_X = 0
OFFSET_Y = 0
RUN_SEQUENCE_ON_START = True

print("Loading AudioLabOverlay() once")
_overlay = AudioLabOverlay()
print("Creating HDMI backend once")
_backend = AudioLabHdmiBackend(_overlay)
_state = AppState()
_cache = make_pynq_static_render_cache()
mirror = HdmiEffectStateMirror(
    overlay=_overlay,
    hdmi_backend=_backend,
    app_state=_state,
    renderer=render_frame_800x480,
    render_cache=_cache,
    theme=THEME,
    variant=VARIANT,
    placement="manual",
    offset_x=OFFSET_X,
    offset_y=OFFSET_Y,
)

class FxControls(object):
    def __init__(self, mirror):
        self.mirror = mirror

    def safe_bypass(self):
        return self.mirror.safe_bypass()

    def basic_clean(self):
        return self.mirror.apply_chain_preset("Basic Clean")

    def noise_gate(self, enabled=True, threshold=25, decay=84, damp=85):
        return self.mirror.set_noise_suppressor_settings(
            enabled=enabled, threshold=threshold, decay=decay, damp=damp)

    def comp(self, enabled=True, threshold=45, ratio=35, response=45, makeup=50):
        return self.mirror.set_compressor_settings(
            enabled=enabled, threshold=threshold, ratio=ratio,
            response=response, makeup=makeup)

    def od(self, enabled=True, drive=35, tone=55, level=65):
        return self.mirror.set_guitar_effects(
            overdrive_on=enabled, overdrive_drive=drive,
            overdrive_tone=tone, overdrive_level=level)

    def dist(self, pedal="tube_screamer", drive=52, tone=60, level=30,
             bias=50, tight=60, mix=100):
        return self.mirror.set_distortion_settings(
            pedal=pedal, drive=drive, tone=tone, level=level,
            bias=bias, tight=tight, mix=mix)

    def rat(self, enabled=True, drive=55, filt=35, level=85, mix=100):
        return self.mirror.set_guitar_effects(
            rat_on=enabled, rat_drive=drive, rat_filter=filt,
            rat_level=level, rat_mix=mix)

    def amp(self, enabled=True, gain=60, bass=55, mid=50, treble=45,
            presence=45, resonance=35, master=75, character=60):
        return self.mirror.set_guitar_effects(
            amp_on=enabled, amp_input_gain=gain, amp_bass=bass,
            amp_middle=mid, amp_treble=treble, amp_presence=presence,
            amp_resonance=resonance, amp_master=master,
            amp_character=character)

    def cab(self, enabled=True, mix=100, level=100, model=2, air=35):
        return self.mirror.set_guitar_effects(
            cab_on=enabled, cab_mix=mix, cab_level=level,
            cab_model=model, cab_air=air)

    def eq(self, enabled=True, low=50, mid=55, high=60):
        return self.mirror.set_guitar_effects(
            eq_on=enabled, eq_low=low * 2, eq_mid=mid * 2,
            eq_high=high * 2)

    def reverb(self, enabled=True, mix=35, decay=60, tone=65):
        return self.mirror.set_guitar_effects(
            reverb_on=enabled, reverb_decay=decay,
            reverb_tone=tone, reverb_mix=mix)

    def render(self):
        return self.mirror.render(reason="manual")

    def summary(self):
        data = self.mirror.get_state_summary()
        print(json.dumps(data, indent=2, sort_keys=True, default=str))
        return data

    def selected_history(self):
        self.mirror.print_selected_fx_history()
        return list(self.mirror.selected_fx_history)

fx = FxControls(mirror)

def _run_step(step, operation, expected, callback):
    print("[{0:02d}] {1}".format(step, operation))
    try:
        callback()
        mirror.assert_selected_fx(expected)
        actual = mirror.get_selected_fx_actual()
        result = "PASS"
    except Exception as exc:
        actual = mirror.get_selected_fx_actual()
        result = "FAIL: {0!r}".format(exc)
    info = mirror.last_render_info or {}
    print("expected SELECTED FX: {0}".format(expected))
    print("actual SELECTED FX  : {0}".format(actual))
    print("result              : {0}".format(result))
    print("render/copy         : {0:.3f}s / {1}".format(
        info.get("render_s") or 0.0, info.get("framebuffer_copy_s")))
    print("VDMA error bits     : {0}".format(info.get("hdmi_errors")))
    print("")
    if result != "PASS":
        raise AssertionError(result)
    if HOLD_SECONDS_PER_STEP > 0:
        time.sleep(HOLD_SECONDS_PER_STEP)

print("Initial 800x480 HDMI GUI render at x=0 y=0")
mirror.render(reason="initial")

if RUN_SEQUENCE_ON_START:
    _sequence = [
        ("fx.safe_bypass", "SAFE BYPASS", fx.safe_bypass),
        ("fx.basic_clean", "PRESET", fx.basic_clean),
        ("fx.noise_gate", "NOISE SUPPRESSOR", fx.noise_gate),
        ("fx.comp", "COMPRESSOR", fx.comp),
        ("fx.od", "OVERDRIVE", fx.od),
        ("fx.dist", "DISTORTION", fx.dist),
        ("fx.rat", "RAT", fx.rat),
        ("fx.amp", "AMP SIM", fx.amp),
        ("fx.cab", "CAB", fx.cab),
        ("fx.eq", "EQ", fx.eq),
        ("fx.reverb", "REVERB", fx.reverb),
    ]
    for _idx, (_operation, _expected, _callback) in enumerate(_sequence, 1):
        _run_step(_idx, _operation, _expected, _callback)

if RETURN_TO_SAFE_BYPASS:
    fx.safe_bypass()

fx.selected_history()
print("User operations:")
print("  fx.amp(gain=60, bass=55, mid=50, treble=45)")
print("  fx.reverb(mix=35, decay=60)")
print("  fx.safe_bypass()")
print("fx, mirror, and _state are ready")

if FINAL_HOLD_SECONDS > 0:
    print("Final HDMI hold: {0} seconds".format(FINAL_HOLD_SECONDS))
    time.sleep(FINAL_HOLD_SECONDS)
